# Pronóstico de la volatilidad diaria del petróleo WTI: AR(1)-GARCH(1,1)-t vs. Random Forest con rezagos

**Proyecto final — Aprendizaje Supervisado · Maestría en Inteligencia Artificial (USFQ)**

- **Integrante:** Darlyn Ludeña
- **Repositorio:** https://github.com/darlamai/proyecto_final_ml_wti.git
- **Datos:** Yahoo Finance, ticker `CL=F` (serie continua de futuros WTI), precios diarios del 04-ene-2010 al 30-oct-2026: precio de cierre (`WTI`), máximo (`WTI_High`) y mínimo (`WTI_Low`) de cada sesión. Se eliminan las sesiones del 20 y 21 de abril de 2020 por presentar precios negativos (vencimiento del contrato de mayo de 2020).
  - **Retornos:** se calculan a partir del precio de cierre como retornos logarítmicos diarios en porcentaje, $r_t = 100 \cdot \ln(P_t / P_{t-1})$. Son la serie sobre la que se realiza el diagnóstico (EDA), se estima el modelo AR(1)-GARCH(1,1)-t y se construyen las variables explicativas del Random Forest.
  - **Volatilidad de Parkinson:** se calcula a partir de los precios máximo y mínimo de cada sesión mediante el estimador de rango de Parkinson, $\sigma^2_{P,t} = \left[100 \cdot \ln(\text{High}_t / \text{Low}_t)\right]^2 / (4 \ln 2)$, expresado en $\%^2$. Es el proxy de la varianza diaria y la variable objetivo con la que se evalúan los pronósticos de ambos modelos.

Esta notebook es el documento principal del proyecto: contiene la introducción, la descripción metodológica y ejecuta, en orden, las notebooks de cada etapa.

| Ítem de la rúbrica | Pts | Dónde se desarrolla |
|---|---|---|
| Introducción | 0.1 | Sección 1 (esta notebook) |
| EDA | 0.6 | `01_preparacion_datos` y `02_eda_retornos` |
| Feature Extraction | 0.25 | Sección 3 (esta notebook) |
| Feature Selection | 0.25 | Sección 4 (esta notebook) y `03_modelos` |
| Pipelines | 0.5 | Sección 5 (esta notebook) y `03_modelos` |
| Optimización de hiperparámetros | 3.5 | `03_modelos` |
| Comparación estadística de 2 técnicas | 3.5 | `03_modelos` |
| Visualización con t-SNE | 1 | `03_modelos` |
| Idioma | 0.1 | Todo el proyecto |
| Conclusiones | 0.1 | Sección 7 (esta notebook) |
| Referencias | 0.1 | Sección 8 (esta notebook) |


## 1. Introducción

### 1.1 Motivación
El petróleo es uno de los activos más relevantes de la economía mundial y uno de los más volátiles. Sus precios reaccionan a choques de oferta (decisiones de la OPEP+, conflictos geopolíticos), de demanda (recesiones, la pandemia de COVID-19) y a restricciones físicas del mercado, como la capacidad de almacenamiento que llevó al contrato WTI de mayo de 2020 a cerrar en precio negativo.

Para quienes toman decisiones sobre este mercado, anticipar **cuánto** puede moverse el precio suele ser tan importante como anticipar **hacia dónde** se moverá. La volatilidad es un insumo directo para:

- la **gestión de riesgos** (Valor en Riesgo, límites de exposición, requerimientos de margen),
- la valoración de **coberturas y derivados** sobre el crudo,
- la **planificación fiscal y presupuestaria** de economías dependientes del petróleo como la ecuatoriana, cuyos ingresos públicos y balanza comercial son sensibles al precio del crudo, que se cotiza con referencia al WTI.

### 1.2 Problema y pregunta de investigación
Los retornos diarios del petróleo son prácticamente impredecibles a partir de su propio pasado, pero su **volatilidad sí presenta dependencia temporal**: los periodos de alta incertidumbre tienden a agruparse (*volatility clustering*). Desde la econometría, este fenómeno se modela con la familia ARCH/GARCH [1], [2], que sigue siendo una referencia difícil de superar [3]. Desde el aprendizaje automático, modelos no lineales como Random Forest [4] pueden aprender relaciones entre la volatilidad pasada y la futura sin imponer una forma funcional.

> **Pregunta de investigación:** ¿un modelo de machine learning (Random Forest) basado en rezagos de medidas de volatilidad pronostica la volatilidad diaria del WTI con mayor precisión que un modelo econométrico AR(1)-GARCH(1,1) con errores t de Student, y es esa diferencia estadísticamente significativa?

- $H_0$: ambos modelos tienen la misma precisión de pronóstico de la volatilidad del día siguiente.
- $H_1$: la precisión de ambos modelos es distinta.

### 1.3 Planteamiento como problema supervisado
La serie de precios se transforma en retornos logarítmicos diarios $r_t = 100 \cdot \ln(P_t/P_{t-1})$. El problema se plantea como **regresión supervisada**: con la información disponible hasta el día $t$ (entrada $X_t$) se pronostica la varianza del día siguiente (etiqueta $y_{t+1}$). Como la volatilidad no se observa directamente, la etiqueta es un **proxy de la varianza diaria**: _[elegir uno: el retorno al cuadrado $r_{t+1}^2$ / el estimador de rango de Parkinson [5] calculado con los precios máximo y mínimo del día]_.

### 1.4 Enfoque
1. **Preparación y EDA** de la serie: calidad de datos, transformación y diagnóstico de estacionariedad, estacionalidad y dependencia temporal. El diagnóstico (efectos ARCH significativos y colas pesadas) justifica el modelo base.
2. **Modelo base (econométrico):** AR(1)-GARCH(1,1)-t, que entrega un pronóstico de la varianza condicional a un paso.
3. **Modelo de machine learning:** Random Forest con rezagos de $r_t^2$, $|r_t|$ y volatilidades móviles como variables explicativas.
4. **Validación:** validación cruzada para series de tiempo (*walk-forward* con ventana expansiva), 2 repeticiones × 5 folds, dentro del periodo de desarrollo 2010–2024. Se reserva 2025–oct 2026 como test final.
5. **Comparación:** los dos modelos se evalúan en los mismos folds con las funciones de pérdida MSE y QLIKE [6], y se comparan con la prueba de Wilcoxon y la prueba de Diebold-Mariano [7].

### 1.5 Estructura del proyecto
| Notebook | Contenido |
|---|---|
| `01_preparacion_datos` | Calidad de datos, transformación a retornos y división temporal |
| `02_eda_retornos` | Diagnóstico de la serie y elección del modelo base |
| `03_modelos` | Variables, pipelines, optimización, comparación estadística y t-SNE |

## 2. Ejecución de las etapas del proyecto
Cada notebook se ejecuta en orden y comparte sus resultados a través de `data/processed/`.

In [1]:
from pathlib import Path
RAIZ = Path.cwd()
assert (RAIZ / "notebooks").exists(), "Ejecutar esta notebook desde la raíz del proyecto"

In [ ]:
%run ./notebooks/01_preparacion_datos.ipynb
%run ./notebooks/02_eda_retornos.ipynb
%run ./notebooks/03_modelos.ipynb

## 3. Feature Extraction
**No aplica en este proyecto.** La rúbrica define este paso para datos no estructurados (imágenes, texto, audio) cuando se usa *transfer learning*, en cuyo caso un modelo preentrenado extrae representaciones automáticamente.

Los datos de este proyecto son **estructurados**: una serie temporal numérica de precios diarios. Las variables explicativas no se extraen con un modelo preentrenado, sino que se **construyen** a partir de la serie mediante ingeniería de variables basada en conocimiento del dominio (sección 4.1).

## 4. Feature Selection

### 4.1 Variables candidatas
Para cada día $t$ se construyen variables que usan **solo información disponible hasta $t$**, de modo que no haya fuga de información hacia la etiqueta $y_{t+1}$:

| Grupo | Variables | Justificación |
|---|---|---|
| Rezagos de la varianza | $r_t^2, r_{t-1}^2, \dots, r_{t-9}^2$ | Captura el agrupamiento de volatilidad (equivalente no lineal del término ARCH) |
| Rezagos de la magnitud | $\lvert r_t\rvert, \dots, \lvert r_{t-9}\rvert$ | Medida de volatilidad menos sensible a valores extremos |
| Volatilidad móvil | Desviación estándar de $r$ en 5, 21 y 63 días | Volatilidad de corto, mediano y largo plazo (análogo del término GARCH) |
| Retorno con signo | $r_t$ | Efecto apalancamiento: las caídas tienden a elevar más la volatilidad que las subidas |
| _(Si se usa Parkinson)_ | Rezagos del estimador de Parkinson | Medida de volatilidad intradía más eficiente que $r_t^2$ |

### 4.2 Técnica de selección tipo *filter*
Se aplica un método de filtro: cada variable se evalúa por su relación individual con la etiqueta, **independientemente del modelo**, y se conservan las $k$ con mayor puntaje.

- **Criterio:** información mutua (`mutual_info_regression`), que mide dependencia lineal y no lineal entre cada variable y la volatilidad del día siguiente. Es coherente con un modelo no lineal como Random Forest. _(Alternativa: `f_regression`, basada en la correlación lineal.)_
- **Implementación:** `SelectKBest(score_func=mutual_info_regression, k=k)` dentro del pipeline.
- **Número de variables $k$:** se trata como hiperparámetro y se optimiza junto con los del Random Forest.

La selección se ajusta **solo con el entrenamiento de cada fold**. Si se calculara con toda la muestra, la información mutua usaría datos de validación y los resultados serían optimistas.

El modelo AR(1)-GARCH(1,1)-t no usa este paso: su especificación (un rezago en la media y uno en cada término de la varianza) se definió en el EDA a partir de la PACF de $r_t$, la ACF de $r_t^2$ y la prueba ARCH-LM.

## 5. Pipelines
El modelo de machine learning se implementa como un `Pipeline` de scikit-learn, que encadena las transformaciones y el estimador en un solo objeto:

```
Pipeline([
    ("seleccion", SelectKBest(mutual_info_regression, k)),   # filtro (sección 4)
    ("modelo",    TransformedTargetRegressor(
                      regressor=RandomForestRegressor(...),
                      func=np.log, inverse_func=np.exp)),     # entrena sobre log(varianza)
])
```

Usar un pipeline aporta:

1. **Ausencia de fuga de información:** en cada fold, `fit` ajusta la selección de variables y el modelo solo con el entrenamiento, y `predict` aplica esas mismas transformaciones a la validación.
2. **Optimización conjunta:** Optuna explora a la vez los hiperparámetros del filtro ($k$) y del Random Forest (`n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`).
3. **Reproducibilidad y modularidad:** el mismo objeto se reutiliza en la validación cruzada, la curva de aprendizaje, la comparación estadística y la evaluación en el test final.

**Decisiones de diseño:**
- **Sin escalado:** Random Forest se basa en particiones por umbrales, por lo que no le afecta la escala de las variables. Un `StandardScaler` no cambiaría el resultado.
- **Etiqueta en logaritmo:** la varianza diaria es positiva y muy asimétrica (valores extremos en 2020). Entrenar sobre $\ln(y)$ estabiliza el ajuste. `TransformedTargetRegressor` aplica la transformación y la invierte automáticamente al predecir. _(Si el proxy es $r_t^2$, se usa $\ln(y + \epsilon)$ porque puede haber días con retorno cero.)_
- **Modelo base con la misma interfaz:** el AR(1)-GARCH(1,1)-t se envuelve en una clase con métodos `fit` y `predict`, de modo que ambos modelos se evalúan con exactamente el mismo código y los mismos folds.

## 6. Resumen de resultados
_Completar al final: tabla con MSE y QLIKE promedio por modelo en los folds de validación y en el test final, p-valores de Wilcoxon y Diebold-Mariano, y principales hallazgos del t-SNE._

## 7. Conclusiones
_Redactar con base en los resultados (no teoría). Puntos a cubrir:_
- Qué mostró el diagnóstico de la serie y cómo justificó el modelo base.
- Qué modelo pronosticó mejor la volatilidad, en qué periodos y si la diferencia fue significativa.
- Qué variables resultaron más relevantes en la selección y en el Random Forest.
- Resultado en el test final (2025–2026) frente a la validación.
- Limitaciones (proxy de volatilidad, serie continua de futuros, cambios de régimen) y trabajo futuro.

## 8. Referencias
[1] R. F. Engle, "Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation," *Econometrica*, vol. 50, no. 4, pp. 987–1007, 1982.

[2] T. Bollerslev, "Generalized autoregressive conditional heteroskedasticity," *J. Econometrics*, vol. 31, no. 3, pp. 307–327, 1986.

[3] P. R. Hansen y A. Lunde, "A forecast comparison of volatility models: Does anything beat a GARCH(1,1)?," *J. Appl. Econometrics*, vol. 20, no. 7, pp. 873–889, 2005.

[4] L. Breiman, "Random forests," *Mach. Learn.*, vol. 45, no. 1, pp. 5–32, 2001.

[5] M. Parkinson, "The extreme value method for estimating the variance of the rate of return," *J. Business*, vol. 53, no. 1, pp. 61–65, 1980.

[6] A. J. Patton, "Volatility forecast comparison using imperfect volatility proxies," *J. Econometrics*, vol. 160, no. 1, pp. 246–256, 2011.

[7] F. X. Diebold y R. S. Mariano, "Comparing predictive accuracy," *J. Bus. Econ. Statist.*, vol. 13, no. 3, pp. 253–263, 1995.

_Agregar: referencias del software (scikit-learn, arch, statsmodels, Optuna, yfinance) y de cualquier otra fuente usada. Si se usó un asistente de IA para generar código, colocar los prompts como comentarios de Python en la celda correspondiente._